<a href="https://colab.research.google.com/github/bribibel/clustal_comparer/blob/main/clustal_alignment_comparer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clustal Alignment Sequence comparer

This program will take an amino acid alignment in clustal format and allow you to
1) Scroll through the alignment, colored by property - hover over a location to show identity & number
2) Enter a specific amino acid position in a selected sequence and display the corresponding amino acid and position number in the other sequences
3) Find positions of difference between selected sequences
4) Find positions of difference between groups of sequences (places where the amino acid is the same at that position within sets but different between sets)

To use . . .

1. Run the cells in order.
2. Cell 2 will prompt you to **upload your Clustal alignment file** (`.aln`, `.clustal`, `.txt`).

First, it will install needed programs

In [1]:
# Install/import widget support (ipywidgets ships with Colab already)
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import re

Now, it will take your alignment in clustal format (which you can get through UniProt's Align function, etc.)

In [2]:
# Upload your alignment file
from google.colab import files

uploaded = files.upload()  # choose your .aln / .clustal / .txt file
FILENAME = next(iter(uploaded))
print(f"Loaded: {FILENAME}")


Saving BsafBsub_MDH_LDH.clustal to BsafBsub_MDH_LDH.clustal
Loaded: BsafBsub_MDH_LDH.clustal


In [3]:
# Parsing functions

def parse_clustal(path):
    """Parse a CLUSTAL(-like) alignment file.

    Returns (order, sequences) where `order` is a list of sequence names
    in the order they first appear, and `sequences` is a dict mapping
    name -> full aligned sequence string (gaps included).
    """
    order = []
    chunks = {}

    with open(path, "r") as fh:
        lines = fh.readlines()

    seq_chunk_re = re.compile(r"[A-Za-z\-\.]+")

    for raw in lines:
        line = raw.rstrip("\n")

        if not line.strip():
            continue
        if line.upper().startswith("CLUSTAL"):
            continue
        # Consensus/conservation lines are usually indented and contain
        # symbols like *, :, . rather than a name + sequence chunk.
        if line[0].isspace():
            continue

        parts = line.split()
        if len(parts) < 2:
            continue

        name, seq = parts[0], parts[1]
        if not seq_chunk_re.fullmatch(seq):
            continue

        if name not in chunks:
            chunks[name] = []
            order.append(name)
        chunks[name].append(seq)

    sequences = {name: "".join(parts) for name, parts in chunks.items()}

    lengths = {len(s) for s in sequences.values()}
    if len(lengths) > 1:
        raise ValueError(
            f"Parsed sequences have unequal lengths {lengths}; "
            "the file may not be a valid CLUSTAL alignment."
        )

    return order, sequences


def build_column_maps(sequences):
    """For each sequence build:
       col_to_resnum[name][col] -> residue number (1-based) or None if gap
       resnum_to_col[name][resnum] -> alignment column (0-based)
    """
    col_to_resnum = {}
    resnum_to_col = {}
    for name, seq in sequences.items():
        c2r = []
        r2c = {}
        resnum = 0
        for col, ch in enumerate(seq):
            if ch in "-.":
                c2r.append(None)
            else:
                resnum += 1
                c2r.append(resnum)
                r2c[resnum] = col
        col_to_resnum[name] = c2r
        resnum_to_col[name] = r2c
    return col_to_resnum, resnum_to_col


order, sequences = parse_clustal(FILENAME)
col_to_resnum, resnum_to_col = build_column_maps(sequences)
print(f"Parsed {len(order)} sequences, alignment length {len(sequences[order[0]])}")


Parsed 4 sequences, alignment length 326


## Scrollable alignment view

A colored, monospace view of the full alignment with a position ruler on
top. It's in a fixed-height box so you can scroll through long alignments
without the notebook output growing forever (scrolls vertically for many
sequences, and horizontally for long alignments).


In [4]:
# Scrollable, colored alignment viewer (hover over a residue to see its number)

import html as _html

# Rough Clustal-style coloring by residue chemistry
AA_COLORS = {
    "A": "#80a0f0", "I": "#80a0f0", "L": "#80a0f0", "M": "#80a0f0",
    "F": "#80a0f0", "W": "#80a0f0", "V": "#80a0f0",           # hydrophobic - blue
    "K": "#f01505", "R": "#f01505",                            # positive - red
    "D": "#c048c0", "E": "#c048c0",                             # negative - magenta
    "S": "#15a4a4", "T": "#15a4a4", "N": "#15a4a4", "Q": "#15a4a4",  # polar - teal
    "C": "#f08080",                                              # cysteine - pink
    "G": "#f09048",                                              # glycine - orange
    "P": "#c0c000",                                              # proline - yellow
    "H": "#15a4a4", "Y": "#15a4a4",                              # aromatic polar - teal
}


def build_ruler(aln_len):
    ruler_chars = [" "] * aln_len
    for i in range(10, aln_len + 1, 10):
        s = str(i)
        start = i - len(s)
        for j, ch in enumerate(s):
            idx = start + j
            if 0 <= idx < aln_len:
                ruler_chars[idx] = ch
    return "".join(ruler_chars)


def ruler_html(ruler):
    """Each ruler character gets a hover tooltip showing its column number."""
    out = []
    for col, ch in enumerate(ruler):
        title = f"Column {col + 1}"
        display_ch = "&nbsp;" if ch == " " else _html.escape(ch)
        out.append(
            f'<span title="{_html.escape(title)}" style="cursor:default">{display_ch}</span>'
        )
    return "".join(out)


def colorize(seq, name, resnums):
    """Each residue gets a hover tooltip showing sequence name, column, and
    that sequence's own residue number (or 'gap')."""
    out = []
    for col, ch in enumerate(seq):
        resnum = resnums[col]
        if resnum is None:
            title = f"{name} | column {col + 1} | gap"
        else:
            title = f"{name} | column {col + 1} | residue {ch}{resnum}"
        color = AA_COLORS.get(ch.upper())
        style = f"background-color:{color};color:#fff;" if color else ""
        out.append(
            f'<span title="{_html.escape(title)}" style="{style}cursor:default">'
            f'{_html.escape(ch)}</span>'
        )
    return "".join(out)


def render_alignment_html(order, sequences, col_to_resnum, box_height="320px"):
    name_width = max(len(n) for n in order) + 2
    aln_len = len(sequences[order[0]])
    ruler = build_ruler(aln_len)

    lines = [" " * name_width + ruler_html(ruler)]
    for name in order:
        padded_name = _html.escape(name).ljust(name_width).replace(" ", "&nbsp;")
        lines.append(padded_name + colorize(sequences[name], name, col_to_resnum[name]))

    body = "<br>".join(lines)

    legend_items = [
        ("Hydrophobic (A,I,L,M,F,W,V)", "#80a0f0"),
        ("Positive (K,R)", "#f01505"),
        ("Negative (D,E)", "#c048c0"),
        ("Polar (S,T,N,Q,H,Y)", "#15a4a4"),
        ("Cysteine (C)", "#f08080"),
        ("Glycine (G)", "#f09048"),
        ("Proline (P)", "#c0c000"),
    ]
    legend_html = " &nbsp; ".join(
        f'<span style="background-color:{color};color:#fff;padding:1px 5px;'
        f'border-radius:3px;font-size:11px">{label}</span>'
        for label, color in legend_items
    )

    return f"""
    <div style="margin-bottom:6px">{legend_html}</div>
    <div style="margin-bottom:4px;font-size:11px;color:#666">
        Hover over any position (ruler or residue) to see its column / residue number.
    </div>
    <div style="overflow:auto; white-space:pre; font-family:'Courier New', monospace;
                font-size:13px; line-height:1.4; height:{box_height};
                border:1px solid #ccc; border-radius:4px; padding:10px; background:#fafafa;">
{body}
    </div>
    """


alignment_view = widgets.HTML(render_alignment_html(order, sequences, col_to_resnum))
display(alignment_view)


HTML(value='\n    <div style="margin-bottom:6px"><span style="background-color:#80a0f0;color:#fff;padding:1px …

## Position comparer

Use the dropdown menu to pick a reference sequence and a position box to pick a residue number. Click Show to see the corresponding amino acid and residue number for every sequence at that alignment position.

In [5]:
# Interactive viewer: dropdown + position box + Show button

seq_dropdown = widgets.Dropdown(
    options=order,
    description="Sequence:",
    style={"description_width": "initial"},
)

def _max_pos(name):
    r2c = resnum_to_col[name]
    return max(r2c) if r2c else 1

pos_input = widgets.BoundedIntText(
    value=1,
    min=1,
    max=_max_pos(order[0]),
    description="Position:",
    style={"description_width": "initial"},
)

show_btn = widgets.Button(description="Show", button_style="primary")
out = widgets.Output()

def update_pos_bounds(change=None):
    name = seq_dropdown.value
    m = _max_pos(name)
    pos_input.max = m
    if pos_input.value > m:
        pos_input.value = m

seq_dropdown.observe(update_pos_bounds, names="value")

def show_position(b=None):
    with out:
        clear_output()
        ref_name = seq_dropdown.value
        pos = pos_input.value
        r2c = resnum_to_col[ref_name]

        if pos not in r2c:
            print(f"Position {pos} is out of range for {ref_name}.")
            return

        col = r2c[pos]
        ref_residue = sequences[ref_name][col]
        print(f"Alignment column {col + 1}   |   {ref_name} residue {pos} = {ref_residue}\n")

        rows = []
        for name in order:
            residue = sequences[name][col]
            resnum = col_to_resnum[name][col]
            if residue in "-.":
                rows.append({"Sequence": name, "Amino Acid": "-", "Residue #": "(gap)"})
            else:
                rows.append({"Sequence": name, "Amino Acid": residue, "Residue #": resnum})

        df = pd.DataFrame(rows)

        def highlight_ref(row):
            return ["background-color: #dbeeff" if row["Sequence"] == ref_name else "" for _ in row]

        display(df.style.apply(highlight_ref, axis=1))

show_btn.on_click(show_position)
update_pos_bounds()

display(widgets.HBox([seq_dropdown, pos_input, show_btn]))
display(out)


Output()

## Compare sequences and list differences

Pick two or more sequences below (Ctrl/Cmd-click, or Shift-click, to select
multiple in the box) and click **Find Differences** to list every alignment
column where they don't all agree. Each differing position shows the amino
acid + that sequence's own residue number (or `-` for a gap).


In [6]:
# Multi-sequence difference finder

compare_select = widgets.SelectMultiple(
    options=order,
    value=tuple(order),
    description="Compare:",
    rows=min(len(order), 8),
    style={"description_width": "initial"},
)

ignore_gaps_cb = widgets.Checkbox(
    value=False,
    description="Ignore gap-only differences (only flag real substitutions)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

diff_btn = widgets.Button(description="Find Differences", button_style="warning")
diff_out = widgets.Output()

def find_differences(b=None):
    with diff_out:
        clear_output()
        names = list(compare_select.value)

        if len(names) < 2:
            print("Select at least two sequences to compare.")
            return

        aln_len = len(sequences[names[0]])
        rows = []

        for col in range(aln_len):
            residues = {name: sequences[name][col] for name in names}
            non_gap = {n: r for n, r in residues.items() if r not in "-."}

            if ignore_gaps_cb.value:
                # Skip columns where we can't make a real amino-acid comparison
                # (fewer than 2 non-gap residues), and skip columns where all
                # non-gap residues agree.
                if len(non_gap) < 2 or len(set(non_gap.values())) <= 1:
                    continue
            else:
                # Any disagreement counts, including a residue vs. a gap.
                if len(set(residues.values())) <= 1:
                    continue

            row = {"Column": col + 1}
            for name in names:
                r = residues[name]
                if r in "-.":
                    row[name] = "-"
                else:
                    row[name] = f"{r}{col_to_resnum[name][col]}"
            rows.append(row)

        if not rows:
            print(f"No differences found among {', '.join(names)} "
                  f"across {aln_len} alignment columns.")
            return

        df = pd.DataFrame(rows)
        print(f"Found {len(rows)} differing position(s) out of {aln_len} "
              f"alignment columns, comparing: {', '.join(names)}\n")
        display(df)

diff_btn.on_click(find_differences)

display(compare_select)
display(ignore_gaps_cb)
display(diff_btn)
display(diff_out)


SelectMultiple(description='Compare:', index=(0, 1, 2, 3), options=('BsafMDH', 'BsubMDH', 'BsafLDH', 'BsubLDH'…

Checkbox(value=False, description='Ignore gap-only differences (only flag real substitutions)', layout=Layout(…

Button(button_style='warning', description='Find Differences', style=ButtonStyle())

Output()

## Find differences *between sets* of sequences

This looks for **diagnostic / group-specific positions**: alignment
columns where every sequence *within* a group agrees with each other, but
the groups disagree *between* each other (e.g. "all my mammal sequences
have K here, all my bird sequences have R here").

1. Set the number of groups and click **Set Up Groups**.
2. Assign sequences to each group's box (Ctrl/Cmd-click or Shift-click for
   multiple). Leave a sequence out of every box if you don't want it
   included in the comparison. Each sequence should go in exactly one group.
3. Click **Find Group-Specific Differences**.

A column only shows up in the results if each group is internally
consistent at that position — if a group is split (some members have one
residue, others another), that position is skipped rather than guessed at.


In [7]:
# Group-vs-group differential position finder

def find_group_differences(groups, ignore_gaps=False):
    """groups: dict of group_name -> list of sequence names.

    Returns a list of row-dicts, one per alignment column where every group
    is internally conserved (same residue across all its members) but at
    least two groups disagree with each other.
    """
    names_all = [n for members in groups.values() for n in members]
    aln_len = len(sequences[names_all[0]])
    rows = []

    for col in range(aln_len):
        consensus = {}
        skip = False

        for gname, members in groups.items():
            residues = [sequences[n][col] for n in members]

            if ignore_gaps:
                non_gap = [r for r in residues if r not in "-."]
                if not non_gap:
                    consensus[gname] = "GAP"
                    continue
                if len(set(non_gap)) > 1:
                    skip = True   # group disagrees internally -> not usable
                    break
                consensus[gname] = non_gap[0]
            else:
                if len(set(residues)) > 1:
                    skip = True
                    break
                consensus[gname] = residues[0]

        if skip:
            continue
        if len(set(consensus.values())) <= 1:
            continue  # all groups agree -> not a differential position

        row = {"Column": col + 1}
        for gname in groups:
            row[f"{gname} (consensus)"] = consensus[gname]
        for gname, members in groups.items():
            for n in members:
                r = sequences[n][col]
                row[n] = "-" if r in "-." else f"{r}{col_to_resnum[n][col]}"
        rows.append(row)

    return rows


# --- UI ---

num_groups_input = widgets.BoundedIntText(
    value=2, min=2, max=8, description="Number of groups:",
    style={"description_width": "initial"},
)
build_groups_btn = widgets.Button(description="Set Up Groups")
groups_container = widgets.VBox([])
group_selectors = []


def build_group_selectors(b=None):
    global group_selectors
    n = num_groups_input.value
    group_selectors = []
    boxes = []
    for i in range(n):
        sel = widgets.SelectMultiple(
            options=order,
            description=f"Group {i + 1}:",
            rows=min(len(order), 6),
            style={"description_width": "initial"},
        )
        group_selectors.append(sel)
        boxes.append(sel)
    groups_container.children = boxes


build_groups_btn.on_click(build_group_selectors)
build_group_selectors()  # initialize with the default number of groups

group_ignore_gaps_cb = widgets.Checkbox(
    value=False,
    description="Ignore gap-only differences (skip pure indel positions)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

find_group_diff_btn = widgets.Button(
    description="Find Group-Specific Differences", button_style="success"
)
group_diff_out = widgets.Output()


def find_group_differences_cb(b=None):
    with group_diff_out:
        clear_output()

        groups = {}
        for i, sel in enumerate(group_selectors):
            members = list(sel.value)
            if members:
                groups[f"Group {i + 1}"] = members

        if len(groups) < 2:
            print("Assign sequences to at least two groups (each with at least one sequence).")
            return

        seen = {}
        overlap_found = False
        for gname, members in groups.items():
            for m in members:
                if m in seen:
                    print(
                        f"Warning: '{m}' is assigned to both {seen[m]} and {gname}. "
                        "Each sequence should belong to only one group."
                    )
                    overlap_found = True
                seen[m] = gname
        if overlap_found:
            return

        rows = find_group_differences(groups, ignore_gaps=group_ignore_gaps_cb.value)

        if not rows:
            print("No group-specific differential positions found.")
            return

        df = pd.DataFrame(rows)
        print(
            f"Found {len(rows)} position(s) where each group is internally "
            f"conserved but differs from the other group(s):\n"
        )
        display(df)


find_group_diff_btn.on_click(find_group_differences_cb)

display(widgets.HBox([num_groups_input, build_groups_btn]))
display(groups_container)
display(group_ignore_gaps_cb)
display(find_group_diff_btn)
display(group_diff_out)


Checkbox(value=False, description='Ignore gap-only differences (skip pure indel positions)', layout=Layout(wid…

Button(button_style='success', description='Find Group-Specific Differences', style=ButtonStyle())

Output()